In [ ]:
import geopandas as gpd
import os

# 1. Carregar o ficheiro de edifícios 
ficheiro_origem = "MstCSCS_Sem_2526.gpkg"

print(f"A carregar o ficheiro: {ficheiro_origem}...")
edificios = gpd.read_file(ficheiro_origem)

# 2. RESOLUÇÃO DE SCR
if edificios.crs is None:
    print("Aviso: O ficheiro não tem SCR definido. A aplicar o sistema oficial de Portugal (EPSG:3763)...")
    edificios = edificios.set_crs("EPSG:3763")

# 3. Descobrir todos os tipos únicos de edifícios que existem na coluna 'Layer'
tipos_unicos = edificios['Layer'].unique()
print(f"\nForam encontrados {len(tipos_unicos)} tipos de edifícios na coluna 'Layer':")
for t in tipos_unicos:
    print(f" - {t}")

# 4. Criar e exportar as camadas organizadas num único GeoPackage
nome_gpkg_final = "Edificios_Divididos_Por_Tipo.gpkg"

# Se o ficheiro final já existir de um teste anterior, removemo-lo para começar do zero
if os.path.exists(nome_gpkg_final):
    os.remove(nome_gpkg_final)

print(f"\nA iniciar a exportação para o ficheiro: {nome_gpkg_final}")

for tipo in tipos_unicos:
    # Isolar apenas as linhas deste tipo específico
    edificios_do_tipo = edificios[edificios['Layer'] == tipo].copy()
    
    # CORREÇÃO DO ERRO: Remover a coluna 'fid' interna para evitar conflitos na gravação
    if 'fid' in edificios_do_tipo.columns:
        edificios_do_tipo = edificios_do_tipo.drop(columns=['fid'])
    
    # Criar um nome de camada limpo (em minúsculas e sem espaços)
    nome_camada = str(tipo).lower().replace(' ', '_').replace('-', '_')
    
    # Guardar a camada dentro do GeoPackage
    edificios_do_tipo.to_file(nome_gpkg_final, layer=nome_camada, driver="GPKG")
    print(f"-> Camada '{nome_camada}' guardada com sucesso! ({len(edificios_do_tipo)} edifícios)")

print("\n!!! Processo concluído com sucesso !!!")
print(f"Pode agora abrir o ficheiro '{nome_gpkg_final}' no QGIS para ver as camadas separadas.")

A carregar o ficheiro: MstCSCS_Sem_2526.gpkg...


c:\Users\gabri\miniforge3\envs\envEOAD_statsPythonR\Lib\site-packages\pyogrio\geopandas.py:275: UserWarning: More than one layer found in 'MstCSCS_Sem_2526.gpkg': 'ed12_polygons_all_with_heights_clean' (default), 'avr_npolicia_pts', 'postal_code_buildings_assigned'. Specify layer parameter to avoid this warning.
  result = read_func(



Foram encontrados 5 tipos de edifícios na coluna 'Layer':
 - ED08_EDIF_PERMANENTE
 - ED10_EDIF_RELIGIOSA
 - ED05_EDIF_ESCOLAR
 - ED07_EDIF_NOTAVEL
 - ED06_EDIF_INDUSTRIAL

A iniciar a exportação para o ficheiro: Edificios_Divididos_Por_Tipo.gpkg
-> Camada 'ed08_edif_permanente' guardada com sucesso! (84309 edifícios)
-> Camada 'ed10_edif_religiosa' guardada com sucesso! (141 edifícios)
-> Camada 'ed05_edif_escolar' guardada com sucesso! (650 edifícios)
-> Camada 'ed07_edif_notavel' guardada com sucesso! (518 edifícios)
-> Camada 'ed06_edif_industrial' guardada com sucesso! (3031 edifícios)

!!! Processo concluído com sucesso !!!
Pode agora abrir o ficheiro 'Edificios_Divididos_Por_Tipo.gpkg' no QGIS para ver as camadas separadas.


## Separar o ficheiro dos 17mil polígonos por tipo de edifício

In [6]:
import geopandas as gpd

# 1. Carregar a sua camada de 17 mil edifícios (a que não tem os tipos)
print("A carregar os 17 mil edifícios...")
edificios_atuais = gpd.read_file("MstCSCS_Sem_2526.gpkg")

# 2. Carregar o ficheiro antigo que CONTÉM as tipologias na coluna 'Layer'
# (Substitua pelo nome exato do ficheiro antigo onde via os tipos "INDUSTRIAL", etc.)
print("A carregar o ficheiro original com as tipologias...")
edificios_com_tipo = gpd.read_file("Tipologia_edificios_Aveiro.shp")

# 3. Garantir o Sistema de Coordenadas (SCR) para o cruzamento funcionar
if edificios_atuais.crs is None:
    edificios_atuais = edificios_atuais.set_crs("EPSG:3763")
if edificios_com_tipo.crs is None:
    edificios_com_tipo = edificios_com_tipo.set_crs("EPSG:3763")

if edificios_com_tipo.crs != edificios_atuais.crs:
    edificios_com_tipo = edificios_com_tipo.to_crs(edificios_atuais.crs)

# 4. Selecionar apenas a geometria e a coluna 'Layer' do ficheiro antigo para não duplicar colunas
edificios_com_tipo = edificios_com_tipo[['geometry', 'Layer']]

# 5. CRUZAMENTO ESPACIAL: O Python vai ver quais edifícios coincidem no mesmo lugar
# e vai copiar a coluna 'Layer' para os seus edifícios atuais
print("A cruzar os dados para recuperar os tipos de edifícios...")
edificios_recuperados = gpd.sjoin(edificios_atuais, edificios_com_tipo, how="left", predicate="intersects")

# Como o sjoin pode criar colunas repetidas de índices, limpamos:
if 'index_right' in edificios_recuperados.columns:
    edificios_recuperados = edificios_recuperados.drop(columns=['index_right'])
if 'fid' in edificios_recuperados.columns:
    edificios_recuperados = edificios_recuperados.drop(columns=['fid'])

# 6. Gravar o resultado num novo GeoPackage único
nome_saida = "MstCSCS_Com_Tipologias.gpkg"
edificios_recuperados.to_file(nome_saida, driver="GPKG")

print(f"\n!!! Concluído com sucesso !!!")
print(f"Foi gerado o ficheiro: '{nome_saida}' com os 17 mil edifícios E a coluna 'Layer' preenchida.")

A carregar os 17 mil edifícios...


c:\Users\gabri\miniforge3\envs\envEOAD_statsPythonR\Lib\site-packages\pyogrio\geopandas.py:275: UserWarning: More than one layer found in 'MstCSCS_Sem_2526.gpkg': 'ed12_polygons_all_with_heights_clean' (default), 'avr_npolicia_pts', 'postal_code_buildings_assigned'. Specify layer parameter to avoid this warning.
  result = read_func(


A carregar o ficheiro original com as tipologias...
A cruzar os dados para recuperar os tipos de edifícios...

!!! Concluído com sucesso !!!
Foi gerado o ficheiro: 'MstCSCS_Com_Tipologias.gpkg' com os 17 mil edifícios E a coluna 'Layer' preenchida.
